Import libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

Load dataset

In [ ]:
file_path = "/content/Loan_Prediction.csv"
df = pd.read_csv("/content/Loan_Prediction.csv")
print("Dataset loaded successfully!")

Dataset loaded successfully!


In [ ]:
# Clean column names
df.columns = df.columns.str.strip()   # remove spaces
df.rename(columns={'loan(yes/no)': 'loan'}, inplace=True)

print(df.columns)  # verify


Index(['Age', 'AnnualIncome(lakhs)', 'CreditScore(300-900)',
       'LoanAmount(lakhs)', 'LoanTerm(years)', 'EmploymentType', 'loan'],
      dtype='object')


Basic checks

In [ ]:
print(df.isnull().sum())

Age                     0
AnnualIncome(lakhs)     0
CreditScore(300-900)    0
 LoanAmount(lakhs)      0
 LoanTerm(years)        0
 EmploymentType         0
 loan(yes/no)           0
dtype: int64


In [ ]:
print(df.duplicated().sum())

0


In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df['EmploymentType'] = le.fit_transform(df['EmploymentType'])


In [ ]:
print(df.dtypes)


Age                       int64
AnnualIncome(lakhs)     float64
CreditScore(300-900)      int64
LoanAmount(lakhs)         int64
LoanTerm(years)           int64
EmploymentType            int64
loan                      int64
dtype: object


In [ ]:
from sklearn.preprocessing import StandardScaler

X = df.drop('loan', axis=1)
y = df['loan']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


Prepare data for KNN

In [ ]:
X = df.drop('loan', axis=1)
y = df['loan']


Scale features

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

Train-test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.3, random_state=42)

Build KNN model

In [ ]:
knn = KNeighborsClassifier(n_neighbors=3)
knn.fit(X_train, y_train)

KNeighborsClassifier(n_neighbors=3)

Predictions

In [ ]:
y_pred = knn.predict(X_test)

print("\n--- Model Accuracy ---")
print(accuracy_score(y_test, y_pred))

print("\n--- Classification Report ---")
print(classification_report(y_test, y_pred))

print("\n--- Confusion Matrix ---")
print(confusion_matrix(y_test, y_pred))


--- Model Accuracy ---
0.6666666666666666

--- Classification Report ---
              precision    recall  f1-score   support

           0       0.50      1.00      0.67         1
           1       1.00      0.50      0.67         2

    accuracy                           0.67         3
   macro avg       0.75      0.75      0.67         3
weighted avg       0.83      0.67      0.67         3


--- Confusion Matrix ---
[[1 0]
 [1 1]]


Predict for new user input

In [ ]:
new_data = pd.DataFrame([[40,9,690,8,10,1]],
                        columns=['Age','AnnualIncome(lakhs)','CreditScore(300-900)',
                                 'LoanAmount(lakhs)','LoanTerm(years)','EmploymentType'])
new_data_scaled = scaler.transform(new_data)
prediction = knn.predict(new_data_scaled)
print("\nPrediction for new user:", "Default" if prediction[0]==1 else "No Default")


Prediction for new user: No Default


No Default = 0 → The customer is expected to repay the loan successfully.

Default = 1 → The customer is expected to fail to repay (high risk).

High‑risk customers  
Customers with lower credit scores, larger requested loans, longer repayment terms, and self-employed status are most likely to default. Lower annual income also increases predicted risk.

What patterns lead to loan default?  
Defaults tend to appear when weaker credit profiles are paired with big loans and extended terms. Self-employed customers stand out as higher-risk due to variable income.

How do credit score and income influence predictions?  
Stronger credit ratings and higher incomes reduce the model’s default probability. These factors help distinguish lower-risk clients from higher-risk applicants.

Suggest banking policies based on model output.  
Adopt stricter approval criteria for borrowers with poor credit or large loan requests, and require additional verification for self-employed applicants. The bank can also offer financial guidance for borderline cases.

Loan amount dominance  
If loan amount is not standardized, it can dominate distance calculations in KNN and overwhelm other variables. Scaling numeric features ensures a balanced comparison.

KNN vs Decision Trees

KNN is easy to implement but very sensitive to feature scaling and slower on larger datasets. Decision Trees are better with mixed data, easier to interpret, and faster at prediction time.

Should KNN be used in real-time loan approval systems?  
Generally not. KNN requires comparing each new application to many examples, which makes it less practical for fast, high-volume approval systems than tree-based or ensemble models.